In [81]:
import re
import json
import html
import cloudscraper
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

scraper = cloudscraper.create_scraper()

In [71]:
# variables

state = 'mg'

In [72]:
state = 'mg'
content = scraper.get(f"https://www.olx.com.br/autos-e-pecas/carros-vans-e-utilitarios/estado-{state}?lis=home_body_search_bar_2020").text

In [118]:
def extrair_datalayer(html_anuncio):
    """Retorna o primeiro objeto window.dataLayer encontrado na página."""
    soup_anuncio = BeautifulSoup(html_anuncio, "html.parser")

    for script_tag in soup_anuncio.find_all("script"):
        script = script_tag.string or script_tag.get_text()
        match = re.search(r"window\.dataLayer\s*=\s*", script)
        if not match:
            continue

        try:
            data_layer, _ = json.JSONDecoder().raw_decode(script[match.end():])
            if isinstance(data_layer, list) and data_layer and isinstance(data_layer[0], dict):
                return data_layer[0].get("page", {})
        except (json.JSONDecodeError, TypeError):
            continue

    return {}


def extrair_preco_fipe(html_anuncio):
    """O preço FIPE está no estado HTML-escapado da página, fora do dataLayer."""
    html_decodificado = html.unescape(html_anuncio)
    match = re.search(
        r'"abuyFipePrice"\s*:\s*\{\s*"fipePrice"\s*:\s*(\d+(?:\.\d+)?)',
        html_decodificado,
    )
    return float(match.group(1)) if match else None


def valor_tabelavel(valor):
    """Converte valores aninhados em texto JSON para manter cada linha tabular."""
    if isinstance(valor, (dict, list)):
        return json.dumps(valor, ensure_ascii=False)
    return valor


soup = BeautifulSoup(content, "html.parser")
linhas_anuncios = []

for card in soup.select("section.olx-adcard"):
    link = card.select_one('a[data-testid="adcard-link"]')
    if not link:
        continue

    url = urljoin("https://www.olx.com.br", link.get("href", ""))
    imagem = card.select_one(".olx-adcard__media img")
    detalhes_card = [
        item.get("aria-label", item.get_text(" ", strip=True))
        for item in card.select(".olx-adcard__detail")
    ]

    linha = {
        "url": url,
        "titulo_card": link.get_text(" ", strip=True),
        "preco_card": (
            float(
                card.select_one(".olx-adcard__price")
                .get_text(" ", strip=True)
                .replace("R$", "")
                .replace(".", "")
                .replace(",", ".")
                .strip()
            )
            if card.select_one(".olx-adcard__price")
            else None
        ),
        "local_card": (
            card.select_one(".olx-adcard__location").get_text(" ", strip=True)
            if card.select_one(".olx-adcard__location") else None
        ),
        "data_card": (
            card.select_one(".olx-adcard__date").get_text(" ", strip=True)
            if card.select_one(".olx-adcard__date") else None
        ),
        "detalhes_card": " | ".join(detalhes_card),
        "imagem_card": imagem.get("src") if imagem else None,
        "erro_coleta": None,
    }

    try:
        resposta = scraper.get(url)
        resposta.raise_for_status()
        html_anuncio = resposta.text
        pagina = extrair_datalayer(html_anuncio)

        # Campos gerais do anúncio: título, preço, descrição, localização,
        # marca, modelo, ano, quilometragem, combustível, câmbio etc.
        ad_detail = pagina.get("adDetail") or {}
        if isinstance(ad_detail, dict):
            # O adDate de adDetail vem incorreto (data em 1970); guardamos
            # o valor original à parte e usamos o timestamp de detail abaixo.
            linha.update({f"anuncio_{chave}": valor_tabelavel(valor)
                          for chave, valor in ad_detail.items()
                          if chave != "adDate"})

        # Use o timestamp Unix de detail.adDate e transforme diretamente
        # a coluna anuncio_adDate em datetime local de São Paulo.
        detail = pagina.get("detail") or {}
        if isinstance(detail, dict):
            timestamp_anuncio = detail.get("adDate")
            try:
                if timestamp_anuncio is not None:
                    timestamp_anuncio = float(timestamp_anuncio)
                    unidade = "ms" if abs(timestamp_anuncio) >= 1e12 else "s"
                    linha["anuncio_adDate"] = pd.to_datetime(
                        timestamp_anuncio, unit=unidade, utc=True
                    ).tz_convert("America/Sao_Paulo")
                else:
                    linha["anuncio_adDate"] = pd.NaT
            except (TypeError, ValueError, OverflowError):
                linha["anuncio_adDate"] = pd.NaT

            referencia = detail.get("abuyPriceRef") or {}
            if isinstance(referencia, dict):
                linha.update({f"referencia_olx_{chave}": valor
                              for chave, valor in referencia.items()})

        # Atributos estruturados do carro (ano, quilometragem, cor,
        # opcionais, documentação e outros que estiverem disponíveis).
        propriedades = pagina.get("adProperties") or []
        if isinstance(propriedades, list):
            for propriedade in propriedades:
                if not isinstance(propriedade, dict):
                    continue
                nome = propriedade.get("name") or propriedade.get("label")
                if nome:
                    linha[f"caracteristica_{nome}"] = valor_tabelavel(
                        propriedade.get("value")
                    )

        # O valor FIPE fica em outro bloco de estado embutido no HTML.
        linha["preco_fipe"] = extrair_preco_fipe(html_anuncio)

    except Exception as erro:
        linha["erro_coleta"] = str(erro)
        linha["preco_fipe"] = None

    linhas_anuncios.append(linha)

# Uma linha por anúncio. Salva CSV para abrir no Excel/LibreOffice.
df_anuncios = pd.DataFrame(linhas_anuncios).drop_duplicates(subset="url")
df_anuncios.to_csv("anuncios_detalhados.csv", index=False, encoding="utf-8-sig")

df_anuncios.head()


,url,titulo_card,preco_card,local_card,data_card,detalhes_card,imagem_card,erro_coleta,anuncio_adId,anuncio_listId,...,caracteristica_condition_report_date,caracteristica_condition_report_detail,caracteristica_condition_report_inspector,caracteristica_condition_report_status,caracteristica_condition_report_visible,preco_fipe,anuncio_financial,caracteristica_financial,anuncio_new_tires,caracteristica_new_tires
0,https://mg.olx.com.br/regiao-de-montes-claros-...,BYD King GL 1.5 16V Aut. (hibrido) 2026,149990.0,"Montes Claros, Ibituruna","Hoje, 10:21",10899 quilômetros rodados | Cor Branco | Motor...,https://img.olx.com.br/thumbs700x500/64/644620...,None,1814802993,1540804516,...,2026-10-02T13:37:13.929Z,A vistoriadora integrada retornou que não exis...,Dekra,UNAVAILABLE,true,151428.0,NaN,NaN,NaN,NaN
1,https://mg.olx.com.br/regiao-de-uberlandia-e-u...,Nissan Kicks SV 1.6 16V Flexstar 5P Aut. 2019,81990.0,"Uberlândia, Patrimônio","Hoje, 10:21",86000 quilômetros rodados | Cor Cinza | Motor ...,https://img.olx.com.br/thumbs700x500/64/643639...,None,1813033881,1539035292,...,NaN,NaN,NaN,NaN,NaN,80669.0,IPVA Pago,IPVA Pago,NaN,NaN
2,https://mg.olx.com.br/belo-horizonte-e-regiao/...,Ford Ecosport Freestyle 1.6 16V Flex 5P 2012,43900.0,"Belo Horizonte, Planalto","Hoje, 10:21",117100 quilômetros rodados | Cor Prata | Motor...,https://img.olx.com.br/thumbs700x500/58/583643...,None,1810690143,1536691309,...,2026-09-29T19:33:14.599Z,A vistoriadora integrada retornou que não exis...,Visão Total,UNAVAILABLE,true,41899.0,IPVA Pago,IPVA Pago,NaN,NaN
3,https://mg.olx.com.br/regiao-de-governador-val...,Toyota Corolla Cross XRE 2.0 16V Flex AUT 2023,133500.0,"Teófilo Otoni, Bela Vista","Hoje, 10:21",41000 quilômetros rodados | Cor Branco | Motor...,https://img.olx.com.br/thumbs700x500/32/323682...,None,1814803009,1540804533,...,2026-10-02T13:27:05.124Z,A vistoriadora integrada retornou que não exis...,Super Visão,UNAVAILABLE,true,133407.0,NaN,NaN,NaN,NaN
4,https://mg.olx.com.br/belo-horizonte-e-regiao/...,Fiat Toro Endurance 1.3 T270 4X2 Flex AUT 2023,98800.0,"Belo Horizonte, Itapoã","Hoje, 10:21",107124 quilômetros rodados | Cor Branco | Moto...,https://img.olx.com.br/thumbs700x500/68/686608...,None,1813647850,1539649297,...,2026-09-28T18:15:06.782Z,A vistoriadora integrada retornou que não exis...,Visão Total,UNAVAILABLE,true,108045.0,IPVA Pago,IPVA Pago,NaN,NaN
